In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA02 - Travel Lodging Entertainment (Public Officials)
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. DATA SOURCE: Uses COUPA FILES ONLY (7 files unioned).
   3. PUBLIC OFFICIAL FILTER: Filters strictly for rows where `PublicOfficial` = 'Y' or 'YES'.
   4. ACCOUNT PARSING: Extracts Cost Center and Category Code directly from the Account string.
   5. TARGET CATEGORY CODES: Hardcoded explicitly to the 29 target codes.
   6. CC MAPPING & THE 793 RULE: Maps to AU_IDs using the bootstrap. Applies the 
      exception rule that Category Code 793 is ONLY valid for AU 101016.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        Account AS Raw_Account,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cost_Center_ID,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Mapped_To_AU AS (
    SELECT 
        c.*,
        m.AU_ID
    FROM Coupa_Parsed c
    INNER JOIN vw_cost_center_mapping_bootstrap m 
        ON c.Cost_Center_ID = CASE WHEN TRIM(m.Cost_Center_ID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(m.Cost_Center_ID), 4), 4, '0') ELSE UPPER(TRIM(m.Cost_Center_ID)) END
),

Flagged_AUs AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Mapped_To_AU
    WHERE UPPER(PublicOfficial) IN ('Y', 'YES')
      AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
      -- THE 793 EXCEPTION RULE
      AND NOT (CatCode = '793' AND AU_ID != '101016')
    GROUP BY AU_ID
)

SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA02' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
WITH Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        Account AS Raw_Account,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cost_Center_ID,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Mapped_To_AU AS (
    SELECT 
        c.*,
        m.AU_ID,
        m.AU_Name
    FROM Coupa_Parsed c
    LEFT JOIN vw_cost_center_mapping_bootstrap m 
        ON c.Cost_Center_ID = CASE WHEN TRIM(m.Cost_Center_ID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(m.Cost_Center_ID), 4), 4, '0') ELSE UPPER(TRIM(m.Cost_Center_ID)) END
)

SELECT 
    Raw_Account,
    Cost_Center_ID,
    CatCode,
    PublicOfficial,
    AU_ID AS Bridged_Master_AU,
    AU_Name AS Bridged_AU_Name,

    -- RULE-BY-RULE CHECKS
    CASE 
        WHEN AU_ID IS NULL THEN 'FAIL: CC failed to bridge to Master AU'
        ELSE 'Pass'
    END AS Bridge_Check,
    CASE 
        WHEN UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES') THEN 'Pass'
        ELSE 'FAIL: Not a Public Official'
    END AS PublicOfficial_Check,
    CASE 
        WHEN CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 'Pass'
        ELSE 'FAIL: Invalid Category Code'
    END AS CatCode_Check,
    CASE 
        WHEN CatCode = '793' AND AU_ID != '101016' THEN 'FAIL: Cat 793 strictly locked to 101016'
        ELSE 'Pass'
    END AS Exception_793_Check,

    -- FINAL DECISION (THIS IS WHAT DRIVES THE EBA02 YES/NO)
    CASE 
        WHEN AU_ID IS NOT NULL
         AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
         AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
         AND NOT (CatCode = '793' AND AU_ID != '101016')
        THEN 'YES'
        ELSE 'NO'
    END AS Final_Decision,
    CASE 
        WHEN AU_ID IS NOT NULL
         AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
         AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
         AND NOT (CatCode = '793' AND AU_ID != '101016')
        THEN CONCAT(
            'YES because AU ', AU_ID,
            ' is bridged, PublicOfficial=', UPPER(TRIM(PublicOfficial)),
            ', CatCode=', CatCode,
            CASE WHEN CatCode = '793' THEN ' (valid only for AU 101016)' ELSE '' END
        )
        ELSE NULL
    END AS Why_Yes,
    CASE 
        WHEN AU_ID IS NULL THEN 'NO because CC failed to bridge to Master AU'
        WHEN UPPER(TRIM(PublicOfficial)) NOT IN ('Y', 'YES') THEN 'NO because row is not PublicOfficial=Y/YES'
        WHEN CatCode NOT IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 'NO because Category Code is outside EBA02 list'
        WHEN CatCode = '793' AND AU_ID != '101016' THEN 'NO because CatCode 793 is valid only for AU 101016'
        ELSE NULL
    END AS Why_No

FROM Mapped_To_AU
-- Optional: Uncomment below to only see rows that contribute to an AU-level YES response
-- WHERE AU_ID IS NOT NULL
--   AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
--   AND CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
--   AND NOT (CatCode = '793' AND AU_ID != '101016')
ORDER BY Cost_Center_ID;